In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date

fake = Faker()
np.random.seed(42)

def generate_cardholder_data(n=1000):
    data = []
    for _ in range(n):
        p = fake.profile()
        balance = np.random.randint(500, 25000)
        
        # Calculate Age
        dob = p['birthdate']
        age = date.today().year - dob.year
        
        # Determine Delinquency (Synthetic Logic)
        prob = (balance / 25000) * 0.4 + (1 - (age / 80)) * 0.3 + np.random.random() * 0.3
        is_delinquent = 1 if prob > 0.75 else 0
        
        data.append({
            'Name': p['name'], 
            'Sex': p['sex'], 
            'Age': age,
            # FIXED: 'job' is the correct key in fake.profile()
            'Occupation': p.get('job', 'Unemployed'), 
            # FIXED: 'residence' is often a full address; cleaning it for city/state
            'Location': p['residence'].replace('\n', ', '),
            'Balance': balance, 
            'Is_Delinquent': is_delinquent
        })
    return pd.DataFrame(data)

# Run the generation
df = generate_cardholder_data()

# Continue with Regression...
le = LabelEncoder()
df['Sex_Code'] = le.fit_transform(df['Sex'])

X = df[['Balance', 'Age', 'Sex_Code']]
y = df['Is_Delinquent']

model = LogisticRegression()
model.fit(X, y)

# Assign Risk Scores
df['Risk_Score'] = model.predict_proba(X)[:, 1]

# Display Result
print(df[['Name', 'Occupation', 'Risk_Score']].sort_values('Risk_Score', ascending=False).head(5))

              Name                           Occupation  Risk_Score
361      Erika May                         Photographer    0.786762
402   John Sanders           Geophysical data processor    0.773564
230    Daniel Todd                         Psychiatrist    0.769739
573  Marcus Fisher                  Pensions consultant    0.755058
767   Megan Rivera  Social research officer, government    0.751704
